# Notebook 2: Three Models of Quantum Computation

This notebook provides conceptual, hands-on examples for the three primary paradigms of quantum computing:
1. **The Circuit Model** (discrete unitary gates).
2. **Measurement-Based Quantum Computing (MBQC)** (adaptive measurements on a cluster state).
3. **Adiabatic Quantum Computing (AQC)** (continuous-time ground-state tracking).

We use Qiskit for the circuit model, PennyLane/Qiskit for cluster states, and NumPy/SciPy for numerical adiabatic evolution.

## Paradigm 1: The Circuit Model

In the circuit model, we initialize qubits, apply a sequence of discrete unitary gates, and measure. Here is a simple 2-qubit circuit to prepare a Bell state $|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$.

In [1]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

# Define a 2-qubit quantum circuit
qc_circuit = QuantumCircuit(2)
qc_circuit.h(0)
qc_circuit.cx(0, 1)

print("Circuit Diagram:")
print(qc_circuit.draw(output='text'))

# Compute exact state vector
sv_circuit = Statevector.from_instruction(qc_circuit)
print("\nState Vector:", sv_circuit.data)
print("Probabilities:", sv_circuit.probabilities_dict())

Circuit Diagram:
     ┌───┐     
q_0: ┤ H ├──■──
     └───┘┌─┴─┐
q_1: ─────┤ X ├
          └───┘

State Vector: [0.70710678+0.j 0.        +0.j 0.        +0.j 0.70710678+0.j]
Probabilities: {np.str_('00'): np.float64(0.4999999999999999), np.str_('11'): np.float64(0.4999999999999999)}


## Paradigm 2: Measurement-Based Quantum Computing (MBQC)

Instead of active gate applications, MBQC prepares a highly entangled **cluster state** beforehand, and performs single-qubit measurements in specific bases to propagate information.

Let's prepare a simple 2-qubit cluster state:
$$|\psi_{12}\rangle = \text{CZ}(|+\rangle \otimes |+\rangle)$$
And show how measuring Q0 in the $X$-basis leaves Q1 in a state that depends on Q0's outcome (teleporting the state with a potential Pauli correction).

In [2]:
# Step 1: Prepare the 2-qubit cluster state CZ (H x H) |00>
qc_cluster = QuantumCircuit(2)
qc_cluster.h([0, 1])
qc_cluster.cz(0, 1)

print("Cluster state preparation circuit:")
print(qc_cluster.draw(output='text'))

# Step 2: Measure Q0 in the X basis (by applying H before measurement)
qc_mbqc = qc_cluster.copy()
qc_mbqc.h(0)

# We inspect the state vector before measurement collapse to analyze both outcomes
sv_mbqc = Statevector.from_instruction(qc_mbqc)
print("\nState Vector in X-basis measured frame for Q0:\n", sv_mbqc.data)

# Outcome 0 on Q0 (corresponds to |+> state of cluster qubit 0):
# leaves Q1 in state |+>
# Outcome 1 on Q0 (corresponds to |-> state of cluster qubit 0):
# leaves Q1 in state |-> (which is Z|+>)

Cluster state preparation circuit:
     ┌───┐   
q_0: ┤ H ├─■─
     ├───┤ │ 
q_1: ┤ H ├─■─
     └───┘   

State Vector in X-basis measured frame for Q0:
 [7.07106781e-01+0.j 2.29934717e-17+0.j 2.29934717e-17+0.j
 7.07106781e-01+0.j]


## Paradigm 3: Adiabatic Quantum Computing (AQC)

In AQC, we continuously change the system's Hamiltonian slowly so that the system stays in the ground state. Let's numerically simulate the time-dependent evolution under a simple interpolation:
$$H(t) = \left(1 - \frac{t}{T}\right)\sigma_x + \frac{t}{T}\sigma_z$$
We will track the final ground-state overlap (fidelity) as a function of the total runtime $T$.

In [3]:
import numpy as np
from scipy.integrate import solve_ivp

# Pauli operators
X = np.array([[0, 1], [1, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)

# Time-dependent Hamiltonian
def H_aqc(t, T):
    s = t / T
    return (1 - s) * X + s * Z

# Target state is the ground state of Z (which is |0> = [1, 0])
target_state = np.array([1, 0], dtype=complex)

# Test different runtimes T
for T in [0.1, 1.0, 5.0, 10.0]:
    # Schrodinger equation: d/dt |psi> = -i H(t) |psi>
    def dy_dt(t, y):
        return -1j * H_aqc(t, T) @ y
    
    # Initial state is ground state of H(0) = X, which is |+> = [1/sqrt(2), 1/sqrt(2)]
    y0 = np.array([1.0, 1.0], dtype=complex) / np.sqrt(2)
    
    # Integrate Schrodinger equation from t=0 to t=T
    sol = solve_ivp(dy_dt, [0, T], y0, t_eval=[T])
    psi_final = sol.y[:, -1]
    
    # Calculate ground state overlap (fidelity)
    fidelity = np.abs(np.vdot(target_state, psi_final))**2
    print(f"Total evolution time T: {T:4.1f} | Final ground-state overlap (Fidelity): {fidelity:.4f}")

Total evolution time T:  0.1 | Final ground-state overlap (Fidelity): 0.5008
Total evolution time T:  1.0 | Final ground-state overlap (Fidelity): 0.5776
Total evolution time T:  5.0 | Final ground-state overlap (Fidelity): 0.9996
Total evolution time T: 10.0 | Final ground-state overlap (Fidelity): 0.9967


## Section 4: Equivalence of the Three Approaches

To demonstrate that these three paradigms are computationally equivalent, let us prepare the exact same quantum state, the plus state $|+\rangle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$, using each of the three models:

1. **Circuit Model**: We apply a discrete Hadamard gate $H$ to the initial state $|0\rangle$.
2. **MBQC**: We prepare a 2-qubit cluster state $|\psi_{12}\rangle = \text{CZ}(|+\rangle \otimes |+\rangle)$ and measure the first qubit (Q0) directly in the $Z$-basis. With measurement outcome $0$ on Q0, the second qubit (Q1) collapses into the state $|+\rangle$.
3. **Adiabatic Quantum Computing (AQC)**: We initialize the system in the ground state of an initial Hamiltonian $H_i = -\sigma_z$ (which is $|0\rangle$), and slowly interpolate to a final Hamiltonian $H_f = -\sigma_x$ (whose ground state is $|+\rangle$).

Below we run all three simulations and compare their output statevectors and overlap (fidelity) with the ideal $|+\rangle$ state.

In [4]:
from qiskit.quantum_info import Statevector
from qiskit import QuantumCircuit
import numpy as np
from scipy.integrate import solve_ivp

# Pauli operators
X = np.array([[0, 1], [1, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)

# --- 1. Circuit Model ---
qc_circ = QuantumCircuit(1)
qc_circ.h(0)
sv_circ = Statevector.from_instruction(qc_circ).data
print("Circuit Model Output Statevector |+>:")
print(sv_circ)

# --- 2. MBQC Model ---
# We prepare the 2-qubit cluster state CZ (H x H)|00>
# and measure Q0 in the Z-basis directly (no H before measurement).
# Under outcome 0 on Q0, Q1 collapses into state |+>.
qc_cluster = QuantumCircuit(2)
qc_cluster.h([0, 1])
qc_cluster.cz(0, 1)

sv_cluster = Statevector.from_instruction(qc_cluster).data
# Extract Q1 components when Q0 is 0 (indices 0 and 2 in Qiskit statevector format)
sv_mbqc_q1 = np.array([sv_cluster[0], sv_cluster[2]])
sv_mbqc_q1 = sv_mbqc_q1 / np.linalg.norm(sv_mbqc_q1)
print("\nMBQC Model Output Statevector (Q1 when Q0 measured 0 in Z-basis) |+>:")
print(sv_mbqc_q1)

# --- 3. AQC Model ---
# Initial Hamiltonian: H(0) = -Z (ground state |0>)
# Final Hamiltonian: H(T) = -X (ground state |+>)
# Interpolation: H(t) = -(1 - t/T) Z - (t/T) X
T_slow = 10.0
def H_interp(t):
    s = t / T_slow
    return - (1 - s) * Z - s * X

def dy_dt_aqc(t, y):
    return -1j * H_interp(t) @ y

# Start in ground state of H_i: |0> = [1, 0]
y_init = np.array([1.0, 0.0], dtype=complex)
sol_aqc = solve_ivp(dy_dt_aqc, [0, T_slow], y_init, t_eval=[T_slow])
sv_aqc = sol_aqc.y[:, -1]
print("\nAdiabatic Quantum Computing Output Statevector |+>:")
print(sv_aqc)

# Calculate overlap (fidelity) with the ideal state |+> = [0.7071, 0.7071]
ideal_plus = np.array([1.0, 1.0]) / np.sqrt(2)
fidelity_circ = np.abs(np.vdot(ideal_plus, sv_circ))**2
fidelity_mbqc = np.abs(np.vdot(ideal_plus, sv_mbqc_q1))**2
fidelity_aqc = np.abs(np.vdot(ideal_plus, sv_aqc))**2

print(f"\nFidelity with ideal |+>:")
print(f"Circuit Model: {fidelity_circ:.6f}")
print(f"MBQC Model   : {fidelity_mbqc:.6f}")
print(f"AQC Model    : {fidelity_aqc:.6f}")

Circuit Model Output Statevector |+>:
[0.70710678+0.j 0.70710678+0.j]

MBQC Model Output Statevector (Q1 when Q0 measured 0 in Z-basis) |+>:
[0.70710678+0.j 0.70710678+0.j]

Adiabatic Quantum Computing Output Statevector |+>:
[-0.17597238+0.673111j   -0.24991001+0.67306316j]

Fidelity with ideal |+>:
Circuit Model: 1.000000
MBQC Model   : 1.000000
AQC Model    : 0.996780
